In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split

In [2]:
DATA_PATH = Path("../data/raw/WineQT.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [3]:
TARGET = "quality"
ID_COL = "Id"

y = df[TARGET]
X = df.drop(columns=[TARGET, ID_COL])

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [5]:
quality_distribution_split = pd.concat(
    [
        y_train.value_counts(normalize=True).sort_index().rename("train_proportion"),
        y_test.value_counts(normalize=True).sort_index().rename("test_proportion"),
    ],
    axis=1,
)

quality_distribution_split

,train_proportion,test_proportion
quality,,
3,0.005470,0.004367
4,0.028446,0.030568
5,0.422319,0.423581
6,0.404814,0.401747
7,0.124726,0.126638
8,0.014223,0.013100


In [15]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

MAIN_SCORING = "rmse"

In [7]:
stage4_extra_trees_baseline = {
    "model": "extra_trees_stage4_baseline",
    "mae_mean": 0.4252,
    "rmse_mean": 0.5883,
    "r2_mean": 0.4632,
}

stage4_extra_trees_baseline

{'model': 'extra_trees_stage4_baseline',
 'mae_mean': 0.4252,
 'rmse_mean': 0.5883,
 'r2_mean': 0.4632}

Шаг 5.7 — ExtraTreesRegressor tuning

In [8]:
extra_trees = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1,
)

extra_trees_param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [None, 8, 12],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt"],
}

Количество комбинаций:

In [9]:
np.prod([len(values) for values in extra_trees_param_grid.values()])

np.int64(36)

Шаг 5.8 — RandomForestRegressor tuning

In [10]:
random_forest = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
)

random_forest_param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [None, 8, 12],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt"],
}

Шаг 5.9 — GradientBoostingRegressor tuning

In [11]:
gradient_boosting = GradientBoostingRegressor(
    random_state=42,
)

gradient_boosting_param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3],
    "min_samples_leaf": [1, 3, 5],
}

In [12]:
np.prod([len(values) for values in gradient_boosting_param_grid.values()])

np.int64(36)

5. GridSearchCV helper
Шаг 5.10 — функция для tuning

In [13]:
def tune_model_grid_search(model, param_grid, X_train, y_train, cv, scoring, main_scoring):
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        refit=main_scoring,
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
        verbose=1,
    )
    
    grid_search.fit(X_train, y_train)
    
    return grid_search

Шаг 5.12 — tune Random Forest

In [16]:
random_forest_search = tune_model_grid_search(
    model=random_forest,
    param_grid=random_forest_param_grid,
    X_train=X_train,
    y_train=y_train,
    cv=cv,
    scoring=scoring,
    main_scoring=MAIN_SCORING,
)

random_forest_search.best_params_

Fitting 5 folds for each of 36 candidates, totalling 180 fits


{'max_depth': None,
 'max_features': 'sqrt',
 'min_samples_leaf': 1,
 'n_estimators': 500}

In [18]:
random_forest_search.best_score_

np.float64(-0.5887706400381889)

Шаг 5.11 — tune Extra Trees

In [19]:
extra_trees_search = tune_model_grid_search(
    model=extra_trees,
    param_grid=extra_trees_param_grid,
    X_train=X_train,
    y_train=y_train,
    cv=cv,
    scoring=scoring,
    main_scoring=MAIN_SCORING,
)

extra_trees_search.best_params_

Fitting 5 folds for each of 36 candidates, totalling 180 fits


{'max_depth': None,
 'max_features': 1.0,
 'min_samples_leaf': 2,
 'n_estimators': 500}

In [20]:
extra_trees_search.best_score_

np.float64(-0.5853727769710803)

Шаг 5.13 — tune Gradient Boosting

In [21]:
gradient_boosting_search = tune_model_grid_search(
    model=gradient_boosting,
    param_grid=gradient_boosting_param_grid,
    X_train=X_train,
    y_train=y_train,
    cv=cv,
    scoring=scoring,
    main_scoring=MAIN_SCORING,
)

gradient_boosting_search.best_params_

Fitting 5 folds for each of 36 candidates, totalling 180 fits


{'learning_rate': 0.05,
 'max_depth': 3,
 'min_samples_leaf': 3,
 'n_estimators': 100}

In [22]:
gradient_boosting_search.best_score_

np.float64(-0.6214996840566044)

7. Collect best CV metrics
Шаг 5.14 — функция для извлечения лучшего результата

In [23]:
def extract_best_cv_metrics(search, model_name):
    best_index = search.best_index_
    cv_results = search.cv_results_
    
    result = {
        "model": model_name,
        "mae_mean": -cv_results["mean_test_mae"][best_index],
        "mae_std": cv_results["std_test_mae"][best_index],
        "rmse_mean": -cv_results["mean_test_rmse"][best_index],
        "rmse_std": cv_results["std_test_rmse"][best_index],
        "r2_mean": cv_results["mean_test_r2"][best_index],
        "r2_std": cv_results["std_test_r2"][best_index],
        "best_params": search.best_params_,
    }
    
    return result

Шаг 5.15 — итоговая таблица tuned models

In [24]:
tuning_results = pd.DataFrame([
    extract_best_cv_metrics(extra_trees_search, "extra_trees_tuned"),
    extract_best_cv_metrics(random_forest_search, "random_forest_tuned"),
    extract_best_cv_metrics(gradient_boosting_search, "gradient_boosting_tuned"),
])

tuning_results = tuning_results.sort_values("rmse_mean").set_index("model")

tuning_results

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,best_params
model,,,,,,,
extra_trees_tuned,0.441369,0.029641,0.585373,0.048273,0.468763,0.024252,"{'max_depth': None, 'max_features': 1.0, 'min_..."
random_forest_tuned,0.450184,0.035767,0.588771,0.056304,0.463393,0.033589,"{'max_depth': None, 'max_features': 'sqrt', 'm..."
gradient_boosting_tuned,0.489292,0.034287,0.621500,0.059178,0.402019,0.037964,"{'learning_rate': 0.05, 'max_depth': 3, 'min_s..."


In [25]:
tuning_results.drop(columns=["best_params"]).round(4)

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
model,,,,,,
extra_trees_tuned,0.4414,0.0296,0.5854,0.0483,0.4688,0.0243
random_forest_tuned,0.4502,0.0358,0.5888,0.0563,0.4634,0.0336
gradient_boosting_tuned,0.4893,0.0343,0.6215,0.0592,0.4020,0.0380


In [26]:
for model_name, params in tuning_results["best_params"].items():
    print(model_name)
    print(params)
    print()

extra_trees_tuned
{'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 2, 'n_estimators': 500}

random_forest_tuned
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 500}

gradient_boosting_tuned
{'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 3, 'n_estimators': 100}



8. Compare with Stage 4 ExtraTrees baseline
Шаг 5.16 — comparison against Stage 4 best

In [27]:
comparison_with_stage4 = tuning_results.copy()

comparison_with_stage4["rmse_delta_vs_stage4_extra_trees"] = (
    comparison_with_stage4["rmse_mean"] - stage4_extra_trees_baseline["rmse_mean"]
)

comparison_with_stage4["mae_delta_vs_stage4_extra_trees"] = (
    comparison_with_stage4["mae_mean"] - stage4_extra_trees_baseline["mae_mean"]
)

comparison_with_stage4["r2_delta_vs_stage4_extra_trees"] = (
    comparison_with_stage4["r2_mean"] - stage4_extra_trees_baseline["r2_mean"]
)

comparison_with_stage4.drop(columns=["best_params"]).round(4)

,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,rmse_delta_vs_stage4_extra_trees,mae_delta_vs_stage4_extra_trees,r2_delta_vs_stage4_extra_trees
model,,,,,,,,,
extra_trees_tuned,0.4414,0.0296,0.5854,0.0483,0.4688,0.0243,-0.0029,0.0162,0.0056
random_forest_tuned,0.4502,0.0358,0.5888,0.0563,0.4634,0.0336,0.0005,0.0250,0.0002
gradient_boosting_tuned,0.4893,0.0343,0.6215,0.0592,0.4020,0.0380,0.0332,0.0641,-0.0612


9. Optional: inspect top grid rows

Если хочешь понять, насколько устойчивый tuning, добавь:

In [28]:
def get_top_search_results(search, model_name, top_n=10):
    results = pd.DataFrame(search.cv_results_)
    
    columns = [
        "mean_test_rmse",
        "std_test_rmse",
        "mean_test_mae",
        "std_test_mae",
        "mean_test_r2",
        "std_test_r2",
        "params",
    ]
    
    top_results = (
        results[columns]
        .sort_values("mean_test_rmse", ascending=False)
        .head(top_n)
        .copy()
    )
    
    top_results["model"] = model_name
    top_results["rmse_mean"] = -top_results["mean_test_rmse"]
    top_results["mae_mean"] = -top_results["mean_test_mae"]
    
    return top_results[
        ["model", "rmse_mean", "std_test_rmse", "mae_mean", "std_test_mae", "mean_test_r2", "std_test_r2", "params"]
    ]

extra_trees_top = get_top_search_results(extra_trees_search, "extra_trees")
random_forest_top = get_top_search_results(random_forest_search, "random_forest")
gradient_boosting_top = get_top_search_results(gradient_boosting_search, "gradient_boosting")

extra_trees_top

,model,rmse_mean,std_test_rmse,mae_mean,std_test_mae,mean_test_r2,std_test_r2,params
3,extra_trees,0.585373,0.048273,0.441369,0.029641,0.468763,0.024252,"{'max_depth': None, 'max_features': 1.0, 'min_..."
2,extra_trees,0.585554,0.047413,0.441296,0.028865,0.468390,0.022531,"{'max_depth': None, 'max_features': 1.0, 'min_..."
1,extra_trees,0.587083,0.049046,0.424212,0.026644,0.465381,0.030699,"{'max_depth': None, 'max_features': 1.0, 'min_..."
25,extra_trees,0.587669,0.052102,0.442671,0.027405,0.464784,0.031180,"{'max_depth': 12, 'max_features': 1.0, 'min_sa..."
27,extra_trees,0.588148,0.050518,0.451100,0.031725,0.463997,0.025401,"{'max_depth': 12, 'max_features': 1.0, 'min_sa..."
0,extra_trees,0.588285,0.047960,0.425218,0.025101,0.463178,0.028018,"{'max_depth': None, 'max_features': 1.0, 'min_..."
24,extra_trees,0.588439,0.052245,0.443323,0.027494,0.463394,0.031289,"{'max_depth': 12, 'max_features': 1.0, 'min_sa..."
26,extra_trees,0.588907,0.050203,0.451527,0.031742,0.462645,0.023804,"{'max_depth': 12, 'max_features': 1.0, 'min_sa..."
7,extra_trees,0.590230,0.054857,0.430191,0.032695,0.460422,0.033682,"{'max_depth': None, 'max_features': 'sqrt', 'm..."
6,extra_trees,0.591330,0.055033,0.430245,0.032885,0.458539,0.032259,"{'max_depth': None, 'max_features': 'sqrt', 'm..."


## Controlled hyperparameter tuning conclusions

### Setup

- Task type: regression.
- Target: `quality`.
- Excluded column: `Id`.
- Feature matrix contains all physicochemical columns except `quality` and `Id`.
- The same train/test split as previous stages was used:
  - `test_size=0.2`;
  - `random_state=42`;
  - `stratify=y`.
- The holdout test set was not evaluated.
- Tuning was performed using 5-fold cross-validation on `X_train` only.
- Main tuning metric: RMSE via `neg_root_mean_squared_error`.

### Models tuned

- `ExtraTreesRegressor`;
- `RandomForestRegressor`;
- `GradientBoostingRegressor`.

### Search spaces

ExtraTreesRegressor:

- `n_estimators`: [300, 500]
- `max_depth`: [None, 8, 12]
- `min_samples_leaf`: [1, 2, 4]
- `max_features`: [1.0, "sqrt"]

RandomForestRegressor:

- `n_estimators`: [300, 500]
- `max_depth`: [None, 8, 12]
- `min_samples_leaf`: [1, 2, 4]
- `max_features`: [1.0, "sqrt"]

GradientBoostingRegressor:

- `n_estimators`: [100, 200]
- `learning_rate`: [0.03, 0.05, 0.1]
- `max_depth`: [2, 3]
- `min_samples_leaf`: [1, 3, 5]

### Best parameters

Write here the best parameters for each tuned model.

### Best CV metrics

Write here the tuned model comparison table.

### Comparison with Stage 4 best model

Stage 4 best model:

- `ExtraTreesRegressor`;
- MAE ≈ 0.4252;
- RMSE ≈ 0.5883;
- R² ≈ 0.4632.

Write here:

- whether tuned Extra Trees improved over the Stage 4 Extra Trees baseline;
- whether Random Forest or Gradient Boosting became competitive;
- whether the improvement is meaningful or marginal.

### Selected model after tuning

Write here which model is the best candidate after Stage 5.

Important: this is still not the final model because the holdout test set has not been evaluated yet.

### Leakage control

- `Id` was excluded from the feature matrix.
- The holdout test set was not evaluated.
- Hyperparameter search was performed only on `X_train`.
- Cross-validation was performed inside `GridSearchCV`.
- No preprocessing was fitted on the full dataset.
- No outliers were removed.
- No target transformation was applied.
- Predictions were not rounded for metric calculation.
- No multiclass or binary framing was used.
- The final model was not saved yet.

## Controlled hyperparameter tuning conclusions

### Setup

- Task type: regression.
- Target: `quality`.
- Excluded column: `Id`.
- Feature matrix contains all physicochemical columns except `quality` and `Id`.
- The same train/test split as previous stages was used:
  - `test_size=0.2`;
  - `random_state=42`;
  - `stratify=y`.
- The holdout test set was not evaluated.
- Tuning was performed using 5-fold cross-validation on `X_train` only.
- Main tuning metric: RMSE via `neg_root_mean_squared_error`.

### Models tuned

- `ExtraTreesRegressor`;
- `RandomForestRegressor`;
- `GradientBoostingRegressor`.

### Search spaces

ExtraTreesRegressor:

- `n_estimators`: [300, 500]
- `max_depth`: [None, 8, 12]
- `min_samples_leaf`: [1, 2, 4]
- `max_features`: [1.0, "sqrt"]

RandomForestRegressor:

- `n_estimators`: [300, 500]
- `max_depth`: [None, 8, 12]
- `min_samples_leaf`: [1, 2, 4]
- `max_features`: [1.0, "sqrt"]

GradientBoostingRegressor:

- `n_estimators`: [100, 200]
- `learning_rate`: [0.03, 0.05, 0.1]
- `max_depth`: [2, 3]
- `min_samples_leaf`: [1, 3, 5]

### Best parameters

ExtraTreesRegressor:

- `max_depth`: None
- `max_features`: 1.0
- `min_samples_leaf`: 2
- `n_estimators`: 500

RandomForestRegressor:

- `max_depth`: None
- `max_features`: "sqrt"
- `min_samples_leaf`: 1
- `n_estimators`: 500

GradientBoostingRegressor:

- `learning_rate`: 0.05
- `max_depth`: 3
- `min_samples_leaf`: 3
- `n_estimators`: 100

### Best CV metrics

Tuned ExtraTreesRegressor:

- MAE ≈ 0.4414
- RMSE ≈ 0.5854
- R² ≈ 0.4688

Tuned RandomForestRegressor:

- MAE ≈ 0.4502
- RMSE ≈ 0.5888
- R² ≈ 0.4634

Tuned GradientBoostingRegressor:

- MAE ≈ 0.4893
- RMSE ≈ 0.6215
- R² ≈ 0.4020

### Comparison with Stage 4 best model

Stage 4 best model:

- `ExtraTreesRegressor`;
- MAE ≈ 0.4252;
- RMSE ≈ 0.5883;
- R² ≈ 0.4632.

Compared with the Stage 4 ExtraTrees baseline, tuned ExtraTreesRegressor changed metrics as follows:

- RMSE improved by approximately 0.0029;
- R² improved by approximately 0.0056;
- MAE became worse by approximately 0.0162.

This is not a meaningful improvement. The RMSE gain is very small and smaller than the CV fold variability. Also, the tuned model worsened MAE, which is an important metric for this ordinal regression target.

RandomForestRegressor became competitive by RMSE, but it did not clearly beat the Stage 4 ExtraTrees baseline.

GradientBoostingRegressor did not become competitive in this small search.

### Selected model after tuning

The best tuned model by RMSE is `ExtraTreesRegressor` with:

- `max_depth=None`;
- `max_features=1.0`;
- `min_samples_leaf=2`;
- `n_estimators=500`.

However, because the improvement over the Stage 4 ExtraTrees baseline is marginal and MAE worsened, the selected candidate should be treated cautiously.

For the next stage, the practical candidate can be:

- tuned `ExtraTreesRegressor` if selecting strictly by RMSE;
- Stage 4 default/fixed `ExtraTreesRegressor` if prioritizing simpler model and better MAE.

The final decision should be made before holdout evaluation and clearly documented.

Important: this is still not the final model because the holdout test set has not been evaluated yet.

### Leakage control

- `Id` was excluded from the feature matrix.
- The holdout test set was not evaluated.
- Hyperparameter search was performed only on `X_train`.
- Cross-validation was performed inside `GridSearchCV`.
- No preprocessing was fitted on the full dataset.
- No outliers were removed.
- No target transformation was applied.
- Predictions were not rounded for metric calculation.
- No multiclass or binary framing was used.
- The final model was not saved yet.